### Quantum Computing Assignment - BB84 Simulation (No Attacker)

The aim of the assignment is to simulate the BB84 key distribution protocol.

This notebook is for a simulation of the protocol without an attacker.

In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math
import numpy as np

In [2]:
simulator = BasicSimulator()
NUM_QUBITS = 38

def get_random_bit(sim):
    """Generates a random bit (0 or 1) by measuring a qubit in superposition."""
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    result = sim.run(qc, shots=1, memory=True).result()
    return int(result.get_memory()[0])

def encode_qubits(bits, bases):
    """Encodes bits onto qubits using the specified bases."""
    encoded_qubits = []
    for i in range(len(bits)):
        qc = QuantumCircuit(1, 1)
        if bits[i] == 1:
            qc.x(0)
        if bases[i] == 1:
            qc.h(0)
        encoded_qubits.append(qc)
    return encoded_qubits

def measure_qubits(qubits, bases, sim):
    """Measures qubits using the specified bases."""
    measured_bits = []
    for i in range(len(qubits)):
        qc = qubits[i].copy()
        if bases[i] == 1:
            qc.h(0)
        qc.measure(0, 0)
        result = sim.run(qc, shots=1, memory=True).result()
        measured_bits.append(int(result.get_memory()[0]))
    return measured_bits

### Alice's Encoding

In [3]:
alice_bits = [get_random_bit(simulator) for _ in range(NUM_QUBITS)]
alice_bases = [get_random_bit(simulator) for _ in range(NUM_QUBITS)]

print(f"Alice's bits:  {np.array(alice_bits)}")
print(f"Alice's bases: {np.array(alice_bases)}")

alice_qubits = encode_qubits(alice_bits, alice_bases)
print("Alice has encoded her qubits and sent them to Bob.")

Alice's bits:  [0 1 0 0 0 1 1 1 0 0 0 0 1 1 0 1 1 1 0 1 0 0 1 0 0 1 1 0 0 1 0 1 0 1 0 0 1
 1]
Alice's bases: [0 1 0 1 1 1 1 1 0 1 1 1 1 0 0 0 1 0 0 1 0 0 0 0 1 1 0 1 0 1 0 1 1 0 0 0 1
 0]
Alice has encoded her qubits and sent them to Bob.


### Bob's Measurement

In [4]:
bob_bases = [get_random_bit(simulator) for _ in range(NUM_QUBITS)]
print(f"Bob's bases:   {np.array(bob_bases)}")

bob_bits = measure_qubits(alice_qubits, bob_bases, simulator)
print(f"Bob's measured bits: {np.array(bob_bits)}")

Bob's bases:   [0 1 1 0 1 0 1 1 1 0 0 1 1 0 0 0 1 0 0 0 0 1 1 0 1 0 0 1 0 1 0 1 0 1 1 1 0
 1]
Bob's measured bits: [0 1 1 1 0 1 1 1 1 1 1 0 1 1 0 1 1 1 0 0 0 0 1 0 0 0 1 0 0 1 0 1 0 1 0 0 0
 0]


### Sifting and Key Agreement

In [5]:
sifted_key_alice = []
sifted_key_bob = []
for i in range(NUM_QUBITS):
    if alice_bases[i] == bob_bases[i]:
        sifted_key_alice.append(alice_bits[i])
        sifted_key_bob.append(bob_bits[i])

print(f"\n--- Sifting Results ---")
print(f"Alice's sifted key: {np.array(sifted_key_alice)}")
print(f"Bob's sifted key:   {np.array(sifted_key_bob)}")

keys_match = (np.array_equal(sifted_key_alice, sifted_key_bob))
if keys_match:
    print(f"\nSuccess! The keys match. A shared secret key of length {len(sifted_key_alice)} has been established.")
else:
    print("\nError! The sifted keys do not match. This should not happen in a simulation without an eavesdropper.")


--- Sifting Results ---
Alice's sifted key: [0 1 0 1 1 0 1 1 0 1 1 1 0 0 0 0 1 0 0 1 0 1]
Bob's sifted key:   [0 1 0 1 1 0 1 1 0 1 1 1 0 0 0 0 1 0 0 1 0 1]
A shared secret key of length 22 has been established.
